# 5.1 采样策略 / Sampling strategies

从 logits 到下一个 token 的几种选法，以及温度为 0 时输出仍可能不同的原因。

The ways of getting from logits to the next token, and why outputs can still differ at temperature 0.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现 temperature、top-k 和 top-p / Implement temperature, top-k and top-p

实现三种对 logits 的处理以及把它们串起来的采样函数。测试会把结果与 Hugging Face 的官方实现逐个元素比较，并检查一个可以手算的例子。

Implement the three operations on logits and the sampling function that chains them. The tests compare the results element by element with the official Hugging Face implementation and check a case that can be worked out by hand.

In [ ]:
import torch

# Every function takes the logits of one position, a float tensor (V,), and returns new logits (V,).
# A token is removed by setting its logit to -inf, so that softmax gives it probability 0.


def apply_temperature(logits, temperature):
    """Divide the logits by the temperature (assumed > 0)."""
    # TODO 1
    raise NotImplementedError


def top_k_filter(logits, k):
    """Keep the k largest logits and set all others to -inf."""
    # TODO 2: torch.topk(logits, k).values[-1] is the k-th largest logit
    raise NotImplementedError


def top_p_filter(logits, p):
    """Nucleus sampling: keep the smallest set of top tokens whose total probability is at least p.

    Sort the tokens by probability, largest first. A token is kept if the probability of all tokens
    ranked above it is still below p. The top token is therefore always kept.
    """
    # TODO 3: torch.sort(logits, descending=True) gives the sorted logits and the original indices
    # TODO 4: softmax, cumulative sum, decide which sorted positions to remove
    # TODO 5: put the result back in the original order
    raise NotImplementedError


def sample_token(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    """The usual order of operations: temperature, then top-k, then top-p, then draw one token.

    temperature == 0 means greedy decoding: return logits.argmax(-1) directly.
    Otherwise return a 0-dim long tensor drawn with torch.multinomial(..., generator=generator).
    """
    # TODO 6
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
from transformers.generation.logits_process import TemperatureLogitsWarper, TopKLogitsWarper, TopPLogitsWarper

torch.manual_seed(0)
logits = torch.randn(50) * 2

# 1. Temperature rescales, it does not reorder.
assert torch.allclose(apply_temperature(logits, 0.5), logits * 2)
assert apply_temperature(logits, 3.0).argmax() == logits.argmax()
ent = lambda l: -(torch.softmax(l, -1) * torch.log_softmax(l, -1)).sum().item()
assert ent(apply_temperature(logits, 0.5)) < ent(logits) < ent(apply_temperature(logits, 2.0)), "lower temperature, lower entropy"

# 2. Top-k keeps exactly k tokens and agrees with the Hugging Face implementation.
for k in (1, 5, 20):
    mine = top_k_filter(logits, k)
    assert torch.isfinite(mine).sum() == k
    assert torch.equal(mine, TopKLogitsWarper(top_k=k)(None, logits[None])[0])

# 3. Top-p on a case that can be checked by hand.
hand = torch.tensor([0.5, 0.3, 0.15, 0.05]).log()
assert torch.isfinite(top_p_filter(hand, 0.7)).tolist() == [True, True, False, False]      # 0.5 < 0.7, 0.5+0.3 >= 0.7
assert torch.isfinite(top_p_filter(hand, 0.85)).tolist() == [True, True, True, False]
assert torch.isfinite(top_p_filter(hand, 0.1)).tolist() == [True, False, False, False]     # the top token always stays
shuffled = hand[[2, 0, 3, 1]]
assert torch.isfinite(top_p_filter(shuffled, 0.7)).tolist() == [False, True, False, True], "restore the original order"

# 4. Top-p agrees with the Hugging Face implementation on random logits.
for p in (0.3, 0.6, 0.9, 0.95):
    mine = top_p_filter(logits, p)
    assert torch.equal(mine, TopPLogitsWarper(top_p=p)(None, logits[None])[0]), p
    kept = torch.softmax(logits, -1)[torch.isfinite(mine)]
    assert kept.sum() >= p and kept.sum() - kept.min() < p, "the kept set is the smallest one reaching p"
assert torch.allclose(apply_temperature(logits, 0.7), TemperatureLogitsWarper(0.7)(None, logits[None])[0])

# 5. The number of tokens kept by top-p adapts to the distribution; top-k does not.
peaked, flat = torch.cat([torch.tensor([8.0]), torch.linspace(0, -0.5, 49)]), torch.zeros(50)
assert torch.isfinite(top_p_filter(peaked, 0.9)).sum() == 1 and torch.isfinite(top_p_filter(flat, 0.9)).sum() == 45
assert torch.isfinite(top_k_filter(peaked, 10)).sum() == 10

# 6. sample_token
assert sample_token(logits, temperature=0) == logits.argmax()
g = torch.Generator().manual_seed(0)
draws = torch.stack([sample_token(logits, 1.0, top_k=5, generator=g) for _ in range(300)])
assert set(draws.tolist()) <= set(logits.topk(5).indices.tolist()), "top-k = 5 can only ever produce 5 different tokens"
g = torch.Generator().manual_seed(0)
cold = torch.stack([sample_token(logits, 0.1, generator=g) for _ in range(200)])
assert (cold == logits.argmax()).float().mean() > 0.95, "a low temperature is close to greedy"
print("m51_filters: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

top-k 中用 `logits.masked_fill(logits < kth, float('-inf'))`，其中 `kth` 是第 k 大的 logit。

For top-k use `logits.masked_fill(logits < kth, float('-inf'))`, where `kth` is the k-th largest logit.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

top-p 中，排序后某个位置应当被去掉的条件是排在它前面的 token 的概率之和已经达到 `p`，即 `probs.cumsum(-1) - probs >= p`。

For top-p, a sorted position is removed when the tokens ranked above it already have a total probability of at least `p`, that is `probs.cumsum(-1) - probs >= p`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

恢复原来的顺序可以用 `torch.full_like(logits, float('-inf')).scatter(-1, order, sorted_logits)`。

The original order is restored with `torch.full_like(logits, float('-inf')).scatter(-1, order, sorted_logits)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch


def apply_temperature(logits, temperature):
    return logits / temperature


def top_k_filter(logits, k):
    kth = torch.topk(logits, k).values[..., -1, None]            # the k-th largest logit
    return logits.masked_fill(logits < kth, float("-inf"))


def top_p_filter(logits, p):
    sorted_logits, order = torch.sort(logits, descending=True)
    probs = torch.softmax(sorted_logits, dim=-1)
    mass_before = probs.cumsum(-1) - probs                       # probability of all tokens ranked above this one
    sorted_logits = sorted_logits.masked_fill(mass_before >= p, float("-inf"))
    return torch.full_like(logits, float("-inf")).scatter(-1, order, sorted_logits)


def sample_token(logits, temperature=1.0, top_k=None, top_p=None, generator=None):
    if temperature == 0:
        return logits.argmax(-1)
    logits = apply_temperature(logits, temperature)
    if top_k is not None:
        logits = top_k_filter(logits, top_k)
    if top_p is not None:
        logits = top_p_filter(logits, top_p)
    return torch.multinomial(torch.softmax(logits, -1), 1, generator=generator)[0]
```

</details>

## 2. 实现 beam search / Implement beam search

在 GPT-2 上实现不含结束符处理的 beam search。测试会检查宽度为 1 时与贪心解码一致，宽度为 4 时与 `model.generate(num_beams=4)` 的结果一致，并比较两者找到的序列的对数概率。运行后请阅读打印出的文本，注意概率更高的续写是否读起来更好。

Implement beam search without end-of-sequence handling on GPT-2. The tests check that width 1 agrees with greedy decoding, that width 4 agrees with the result of `model.generate(num_beams=4)`, and compare the log-probabilities of the sequences the two find. Read the printed texts afterwards and consider whether the more probable continuations read better.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def sequence_logprob(model, input_ids, prompt_len):
    """Total log-probability of the continuation, log p(x_{prompt_len} ... x_{T-1} | prompt).

    input_ids: (1, T). Returns a Python float.
    """
    # TODO 1: log-softmax of model(input_ids).logits[0, :-1]; row t predicts token t+1
    # TODO 2: pick the log-probability of each actual next token, sum the ones after the prompt
    raise NotImplementedError


@torch.no_grad()
def beam_search(model, input_ids, beam_size, max_new_tokens):
    """Beam search without end-of-sequence handling: every beam grows by exactly max_new_tokens.

    input_ids: (1, T)
    returns (beams, scores): beams (beam_size, T + max_new_tokens), best first;
                             scores (beam_size,), the total log-probability of each continuation.

    Keep `beams` (b, t) and `scores` (b,). Start with the prompt alone and score 0. In each step:
      logp  = log_softmax(model(beams).logits[:, -1])        (b, V)
      total = scores[:, None] + logp                         every beam extended by every token
      take the beam_size largest entries of total.view(-1); entry i belongs to
      beam i // V and token i % V
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
N = 12
show = lambda ids, T: repr(tok.decode(ids[T:]))

rows = []
for prompt in ["The capital of France is", "My favourite food is", "In machine learning, attention is"]:
    ids = tok(prompt, return_tensors="pt").input_ids
    T = ids.shape[1]

    # 1. Beam size 1 is greedy decoding.
    greedy = model.generate(ids, do_sample=False, max_new_tokens=N, pad_token_id=tok.eos_token_id)
    b1, s1 = beam_search(model, ids, 1, N)
    assert b1.shape == (1, T + N) and torch.equal(b1[0], greedy[0]), "beam size 1 must reproduce greedy decoding"

    # 2. The returned score is the log-probability of the continuation.
    assert abs(s1[0].item() - sequence_logprob(model, b1[:1], T)) < 1e-2

    # 3. Beam size 4 agrees with the Hugging Face implementation and is sorted best first.
    b4, s4 = beam_search(model, ids, 4, N)
    hf = model.generate(ids, num_beams=4, do_sample=False, max_new_tokens=N, pad_token_id=tok.eos_token_id)
    assert b4.shape == (4, T + N) and torch.equal(b4[0], hf[0]), "beam size 4 should match model.generate(num_beams=4)"
    assert (s4[:-1] >= s4[1:]).all()
    assert abs(s4[0].item() - sequence_logprob(model, b4[:1], T)) < 1e-2

    # 4. A wider beam never finds a less probable sequence than greedy here.
    assert s4[0] >= s1[0] - 1e-4
    rows.append((prompt, s1[0].item(), show(b1[0], T), s4[0].item(), show(b4[0], T)))

assert any(r[3] > r[1] + 0.5 for r in rows), "on these prompts beam search finds clearly more probable continuations"
print("m51_beam: all tests passed")
for prompt, sg, tg, sb, tb in rows:
    print(f"  {prompt!r}")
    print(f"    greedy  log p = {sg:7.2f}  {tg}")
    print(f"    beam 4  log p = {sb:7.2f}  {tb}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`total = (scores[:, None] + logp).view(-1)` 把所有候选排成一个长度为 `b * V` 的向量，对它做 `topk(beam_size)`。

`total = (scores[:, None] + logp).view(-1)` lines all candidates up in a vector of length `b * V`, on which `topk(beam_size)` is taken.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

由扁平下标还原出 `parent = flat // V` 和 `token = flat % V`，新的候选是 `torch.cat([beams[parent], token[:, None]], dim=1)`。

Recover `parent = flat // V` and `token = flat % V` from the flat index. The new candidates are `torch.cat([beams[parent], token[:, None]], dim=1)`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


@torch.no_grad()
def sequence_logprob(model, input_ids, prompt_len):
    logp = torch.log_softmax(model(input_ids).logits[0, :-1], -1)             # row t predicts token t+1
    picked = logp.gather(-1, input_ids[0, 1:, None])[:, 0]
    return picked[prompt_len - 1:].sum().item()


@torch.no_grad()
def beam_search(model, input_ids, beam_size, max_new_tokens):
    beams = input_ids                                                         # (1, T) at the start
    scores = torch.zeros(1)
    for _ in range(max_new_tokens):
        logp = torch.log_softmax(model(beams).logits[:, -1], -1)              # (b, V)
        V = logp.shape[-1]
        total = (scores[:, None] + logp).view(-1)                             # every beam extended by every token
        scores, flat = total.topk(beam_size)
        parent, token = flat // V, flat % V
        beams = torch.cat([beams[parent], token[:, None]], dim=1)
    return beams, scores
```

</details>

## 3. 复现温度为 0 时的不一致 / Reproduce inconsistency at temperature 0

实现一个同时记录每一步第一名与第二名 logit 差距的贪心解码。测试先确认同一精度下重复运行的结果完全相同，再把同一份权重转换成 16 位浮点数重新运行，找出两者输出分叉的位置。运行后请查看打印的结果，注意分叉处的差距有多小。在 CPU 上以 16 位精度运行较慢，整个测试大约需要半分钟。

Implement greedy decoding that also records the gap between the logits of the first and second candidate at every step. The tests first confirm that repeated runs at the same precision give identical results and then rerun with the same weights converted to 16-bit floating point and locate where the two outputs split. Look at the printed results and note how small the gap is at the split. Running at 16-bit precision on a CPU is slow, and the whole test takes about half a minute.

需要联网下载模型。/ Downloads a model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Greedy decoding has no randomness: the same logits always give the same token.
# The logits themselves, however, are floating-point numbers, and the same model computed in a
# different precision, on different hardware or with a different batch gives slightly different logits.
# This exercise measures when such small differences change the output.


def top2_gap(logits):
    """Difference between the largest and the second largest logit, as a Python float.

    This is the margin by which greedy decoding makes its choice.
    Convert with logits.float() first so that half-precision inputs work too.
    """
    # TODO 1
    raise NotImplementedError


@torch.no_grad()
def greedy_trace(model, input_ids, max_new_tokens):
    """Greedy decoding that also records how close each decision was.

    Returns (tokens, gaps): two lists of length max_new_tokens,
    the chosen token ids (ints) and the top-2 gap of the logits at each step.
    """
    # TODO 2: the loop from module 0.1, plus top2_gap at every step
    raise NotImplementedError


def first_divergence(a, b):
    """Index of the first position at which two equally long token lists differ, or None."""
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 0. Floating-point addition is not associative. This is the root of everything below.
assert (0.1 + 0.2) + 0.3 != 0.1 + (0.2 + 0.3)

assert abs(top2_gap(torch.tensor([1.0, 3.5, 3.0, -2.0])) - 0.5) < 1e-6
assert first_divergence([1, 2, 3], [1, 2, 3]) is None and first_divergence([1, 2, 3], [1, 5, 3]) == 1

tok = AutoTokenizer.from_pretrained("gpt2")
m32 = AutoModelForCausalLM.from_pretrained("gpt2").eval()
m16 = AutoModelForCausalLM.from_pretrained("gpt2").to(torch.float16).eval()      # same weights, half precision
N = 30
prompts = ["Scientists have discovered that", "The weather today is", "In machine learning, attention is", "The capital of France is"]

# 1. Same model, same precision, same machine: greedy decoding repeats exactly.
ids = tok(prompts[0], return_tensors="pt").input_ids
t_a, g_a = greedy_trace(m32, ids, N)
t_b, _ = greedy_trace(m32, ids, N)
assert len(t_a) == N and len(g_a) == N and all(g >= 0 for g in g_a)
assert first_divergence(t_a, t_b) is None, "nothing random happens in greedy decoding"
hf = m32.generate(ids, do_sample=False, max_new_tokens=N, pad_token_id=tok.eos_token_id)[0, ids.shape[1]:].tolist()
assert t_a == hf

# 2. The same weights in half precision give slightly different logits.
with torch.no_grad():
    l32, l16 = m32(ids).logits[0, -1], m16(ids).logits[0, -1].float()
err = (l32 - l16).abs().max().item()
assert 1e-4 < err < 2.0, err

# 3. Whether that changes the text depends on how close the decisions are.
rows, diverged = [], 0
for p in prompts:
    ids = tok(p, return_tensors="pt").input_ids
    t32, g32 = greedy_trace(m32, ids, N)
    t16, _ = greedy_trace(m16, ids, N)
    d = first_divergence(t32, t16)
    diverged += d is not None
    rows.append((p, d, g32[d] if d is not None else min(g32), tok.decode(t32), tok.decode(t16)))
assert diverged >= 1, "at least one of these prompts should diverge between float32 and float16"
print(f"m51_temp0: all tests passed (largest logit difference between float32 and float16 on one prompt: {err:.3f})")
for p, d, gap, a, b in rows:
    if d is None:
        print(f"  {p!r}: identical for {N} tokens, smallest top-2 gap {gap:.3f}")
    else:
        print(f"  {p!r}: diverges at step {d}, where the top-2 gap was {gap:.4f}")
        print(f"      float32: {a!r}")
        print(f"      float16: {b!r}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`torch.topk(logits.float(), 2).values` 给出最大的两个 logit。

`torch.topk(logits.float(), 2).values` gives the two largest logits.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

循环的写法与 0.1 的贪心解码相同，每一步多记录一个差距。

The loop is that of greedy decoding in 0.1, recording one gap more per step.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def top2_gap(logits):
    top = torch.topk(logits.float(), 2).values
    return (top[0] - top[1]).item()


@torch.no_grad()
def greedy_trace(model, input_ids, max_new_tokens):
    tokens, gaps = [], []
    for _ in range(max_new_tokens):
        logits = model(input_ids).logits[0, -1]
        gaps.append(top2_gap(logits))
        next_id = logits.argmax()
        tokens.append(next_id.item())
        input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
    return tokens, gaps


def first_divergence(a, b):
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return i
    return None
```

</details>

## 4. 手算温度与 top-p / Temperature and top-p by hand

在纸上完成，可以使用计算器。
(a) 三个 token 的 logits 是 $(2, 1, 0)$。分别计算 $T = 1$ 和 $T = 0.5$ 时的概率分布。
(b) 证明温度不改变 token 的排序。
(c) 四个 token 的概率是 $(0.5, 0.3, 0.15, 0.05)$。$p = 0.7$ 和 $p = 0.9$ 时 top-p 各保留哪些 token？保留后重新归一化的概率是多少？
(d) 某一步第一名与第二名的 logit 差为 0.004。如果数值误差使每个 logit 最多变化 0.001，贪心的选择会不会改变？最多变化 0.003 时呢？

Work on paper. A calculator may be used.
(a) Three tokens have the logits $(2, 1, 0)$. Compute the probability distributions for $T = 1$ and $T = 0.5$.
(b) Show that temperature does not change the ranking of tokens.
(c) Four tokens have the probabilities $(0.5, 0.3, 0.15, 0.05)$. Which tokens does top-p keep for $p = 0.7$ and for $p = 0.9$, and what are the renormalized probabilities?
(d) At some step the logits of the first and second candidate differ by 0.004. If numerical error changes every logit by at most 0.001, can the greedy choice change? And with at most 0.003?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) $T = 0.5$ 相当于把 logits 变成 $(4, 2, 0)$。

(a) $T = 0.5$ amounts to turning the logits into $(4, 2, 0)$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 比较差距与 $2\varepsilon$。

(d) Compare the gap with $2\varepsilon$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) $T = 1$ 时，$e^2 : e^1 : e^0 = 7.389 : 2.718 : 1$，归一化得 $(0.665, 0.245, 0.090)$。$T = 0.5$ 时，$e^4 : e^2 : e^0 = 54.60 : 7.389 : 1$，归一化得 $(0.867, 0.117, 0.016)$。温度降低后分布更集中。

(b) $T > 0$ 时，$z_i > z_j$ 等价于 $z_i / T > z_j / T$，而指数函数单调递增，所以 $p_i(T) > p_j(T)$。

(c) $p = 0.7$ 时，$0.5 < 0.7$ 而 $0.5 + 0.3 \ge 0.7$，保留前两个，归一化为 $(0.625, 0.375)$。$p = 0.9$ 时，$0.5 + 0.3 = 0.8 < 0.9$ 而 $0.95 \ge 0.9$，保留前三个，归一化为 $(0.526, 0.316, 0.158)$。

(d) 误差不超过 0.001 时，$2\varepsilon = 0.002 < 0.004$，选择一定不变。误差不超过 0.003 时，$2\varepsilon = 0.006 > 0.004$，选择有可能改变，是否改变取决于误差的具体取值。

---

(a) For $T = 1$, $e^2 : e^1 : e^0 = 7.389 : 2.718 : 1$, normalized $(0.665, 0.245, 0.090)$. For $T = 0.5$, $e^4 : e^2 : e^0 = 54.60 : 7.389 : 1$, normalized $(0.867, 0.117, 0.016)$. A lower temperature concentrates the distribution.

(b) For $T > 0$, $z_i > z_j$ is equivalent to $z_i / T > z_j / T$, and the exponential function is increasing, so $p_i(T) > p_j(T)$.

(c) For $p = 0.7$, $0.5 < 0.7$ and $0.5 + 0.3 \ge 0.7$, so the first two are kept, renormalized $(0.625, 0.375)$. For $p = 0.9$, $0.5 + 0.3 = 0.8 < 0.9$ and $0.95 \ge 0.9$, so the first three are kept, renormalized $(0.526, 0.316, 0.158)$.

(d) With an error of at most 0.001, $2\varepsilon = 0.002 < 0.004$ and the choice cannot change. With at most 0.003, $2\varepsilon = 0.006 > 0.004$ and the choice may change, depending on the actual values of the errors.

</details>